# Ocena zapisanych checkpointów U-Neta

Porównujemy Dice po pojedynczych przekrojach oraz po pełnych wolumenach pacjentów. Jedno przejście modelu dostarcza predykcji do wszystkich pięciu metryk.

Bieżący przykład odtwarza wyniki na walidacji dla seeda 43. Walidacja służy do porównywania kolejnych wariantów. Zbiór testowy pozostaje do oceny po zamknięciu eksperymentów i zamrożeniu protokołu.

## 1. Biblioteki, dane i checkpoint

Wczytujemy zapisany podział pacjentów i wybieramy walidację. TIFF-y przetwarzamy tak samo jak w bazowym treningu, bez augmentacji. Model na CPU wczytuje wagi wybranej epoki i przechodzi w tryb `eval()`.

In [1]:
from pathlib import Path
import sys

import numpy as np
from scipy import ndimage

import torch
import pandas as pd
import tifffile
from torch.utils.data import Dataset, DataLoader

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import UNet

In [2]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)

val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()

In [3]:
def preprocess_pair(image, mask):
    image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255
    mask_tensor = torch.from_numpy(mask > 0).float().unsqueeze(0)

    return (image_tensor, mask_tensor)

class MRIDataset(Dataset):
    def __init__(self, pairs_df, project_root, transform=None):
        self.pairs_df = pairs_df
        self.project_root = project_root
        self.transform = transform

    def __len__(self):
        return self.pairs_df.shape[0]

    def __getitem__(self, index):
        row = self.pairs_df.iloc[index]
        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]
        image, mask = preprocess_pair(tifffile.imread(image_path), tifffile.imread(mask_path))

        if self.transform is not None:
            return self.transform(image, mask)
        return image, mask

In [4]:
checkpoint_path = (
    project_root / "outputs" / "checkpoints" / "unet_bce_dice_best_lr3e-4_seed43.pt"
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model = UNet()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Checkpoint: {checkpoint_path.name}, seed: {checkpoint['seed']}, epoka: {checkpoint['epoch']}")

Checkpoint: unet_bce_dice_best_lr3e-4_seed43.pt, seed: 43, epoka: 15


## 2. Predykcje i definicje Dice

Loader zachowuje kolejność przekrojów pacjenta. Model zwraca logity, a `sigmoid(logits) > 0.5` wyznacza maskę binarną. Łączymy paczki po osi przekrojów, otrzymując wolumen o kształcie `(N, 1, H, W)`.

Dice po przekrojach daje jeden wynik na przekrój. Dice 3D sumuje przecięcie i rozmiary masek po wszystkich przekrojach pacjenta, a następnie liczy jeden iloraz. Dwie puste maski dają wynik 1.

In [5]:
@torch.inference_mode()
def predict_patient(model, patient_loader):
    prediction_batches = []
    mask_batches = []

    for images, masks in patient_loader:
        probabilities = torch.sigmoid(model(images))
        prediction_batches.append(probabilities > 0.5)
        mask_batches.append(masks > 0)

    return torch.cat(prediction_batches, dim=0), torch.cat(mask_batches, dim=0)

In [6]:
def dice_per_sample(predictions, targets):
    targets_logic = targets > 0

    targets_sum = targets_logic.sum(dim=(1, 2, 3))
    predictions_sum = predictions.sum(dim=(1, 2, 3))
    intersection = (targets_logic & predictions).sum(dim=(1, 2, 3))

    denominator = targets_sum + predictions_sum

    return torch.where(
        denominator == 0,
        1.0,
        2 * intersection / denominator.clamp_min(1)
    )


def dice3d(preds, masks):
    intersection = (preds & masks).sum().item()
    preds_count = preds.sum().item()
    masks_count = masks.sum().item()

    denominator = preds_count + masks_count

    dice = 1.0 if denominator == 0 else 2 * intersection / denominator

    return (intersection, preds_count, masks_count, dice)

## 3. Największy spójny obszar 3D

Zdejmujemy oś kanału i etykietujemy spójne obszary w całym wolumenie przy sąsiedztwie 6. Etykieta 0 oznacza tło. Zachowujemy największy obszar predykcji i przywracamy oś kanału. Pusta predykcja pozostaje pusta.

Porównujemy obie wersje predykcji z tą samą maską referencyjną. Pozwala to oddzielić jakość modelu od wpływu postprocessingu.

In [7]:
def keep_largest_component(preds):
    labeled, n_components = ndimage.label(preds.squeeze(1).numpy())
    if n_components == 0:
        return preds

    sizes = np.bincount(labeled.ravel())
    largest_label = sizes[1:].argmax() + 1
    return torch.from_numpy(labeled == largest_label).unsqueeze(1)

## 4. Jedno przejście po walidacji

Dla każdego pacjenta sortujemy przekroje po `slice_id` i wykonujemy predykcję w paczkach. W metrykach 2D każdy przekrój ma tę samą wagę. W metrykach 3D najpierw liczymy wynik całego wolumenu, a potem każdy pacjent ma tę samą wagę w średniej.

Na tym etapie korzystamy z 16 pacjentów walidacyjnych i 615 przekrojów, w tym 236 z adnotacją.

In [8]:
def evaluate_patients_3d(model, split_df, project_root):
    patient_results = []
    dice_batches = []
    annotation_batches = []

    for patient_id, patient_rows in split_df.groupby("patient_id"):
        patient_df = patient_rows.sort_values("slice_id")
        dataset = MRIDataset(patient_df, project_root)
        loader = DataLoader(dataset, batch_size=4, shuffle=False)
        preds, masks = predict_patient(model, loader)

        dice_batches.append(dice_per_sample(preds, masks))
        annotation_batches.append(masks.flatten(1).any(dim=1))

        filtered_preds = keep_largest_component(preds)
        patient_results.append({
            "patient_id": patient_id,
            "dice_3d": dice3d(preds, masks)[-1],
            "dice_3d_filtered": dice3d(filtered_preds, masks)[-1],
        })

    return pd.DataFrame(patient_results), torch.cat(dice_batches), torch.cat(annotation_batches)

In [9]:
patient_results, all_dice, all_annotations = evaluate_patients_3d(model, val_df, project_root)
print(
    f"Pacjenci: {len(patient_results)}, przekroje: {all_dice.numel()}, "
    f"z adnotacją: {all_annotations.sum().item()}"
)

metrics = pd.Series({
    "Dice po wszystkich przekrojach": all_dice.mean().item(),
    "Dice przekrojów z adnotacją": all_dice[all_annotations].mean().item(),
    "Dice przekrojów z pustą maską": all_dice[~all_annotations].mean().item(),
    "Dice 3D po pacjentach": patient_results["dice_3d"].mean(),
    "Dice 3D po pacjentach po filtrowaniu": patient_results["dice_3d_filtered"].mean(),
}, name="Walidacja")
metrics.to_frame().round(4)

Pacjenci: 16, przekroje: 615, z adnotacją: 236


,Walidacja
Dice po wszystkich przekrojach,0.5221
Dice przekrojów z adnotacją,0.4835
Dice przekrojów z pustą maską,0.5462
Dice 3D po pacjentach,0.5034
Dice 3D po pacjentach po filtrowaniu,0.5626


## 5. Wyniki poszczególnych pacjentów

Średni Dice 3D dla seeda 43 rośnie po filtrowaniu z 0,5034 do 0,5626. U pierwszego pacjenta w tabeli spada z 0,2552 do 0,1586, ponieważ część poprawnie zaznaczonych pikseli znalazła się poza największym obszarem. Z tego powodu raportujemy obie wersje.

In [10]:
patient_results.round(4)

,patient_id,dice_3d,dice_3d_filtered
0,TCGA_CS_5396_20010302,0.2552,0.1586
1,TCGA_CS_6665_20010817,0.3499,0.4991
2,TCGA_DU_5849_19950405,0.7754,0.9236
3,TCGA_DU_6407_19860514,0.7899,0.9024
4,TCGA_DU_7013_19860523,0.4528,0.4796
5,TCGA_DU_7304_19930325,0.7102,0.7227
6,TCGA_DU_8164_19970111,0.5582,0.6097
7,TCGA_DU_A5TP_19970614,0.4162,0.5445
8,TCGA_DU_A5TU_19980312,0.6668,0.7543
9,TCGA_FG_6691_20020405,0.7250,0.8907
